In [15]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import numpy as np
from sklearn.model_selection import KFold

device = "cuda" if torch.cuda.is_available() else "cpu"

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_data = datasets.CIFAR10(root="../datasets", train=True, download=True, transform=transform)
test_data  = datasets.CIFAR10(root="../datasets", train=False, download=True, transform=transform)

class SmallCNN(nn.Module):
    def __init__(self, channels=32, depth=2):
        super().__init__()
        layers = []
        in_c = 3

        for _ in range(depth):
            layers += [
                nn.Conv2d(in_c, channels, 3, padding=1),
                nn.ReLU(),
                nn.MaxPool2d(2)
            ]
            in_c = channels
        
        self.features = nn.Sequential(*layers)

        with torch.no_grad():
            dummy = torch.zeros(1, 3, 32, 32)
            out = self.features(dummy)
            self.flatten_dim = out.numel()

        self.classifier = nn.Linear(self.flatten_dim, 10)

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)
    
class BiggerCNN(nn.Module):
    def __init__(self, channels=64, depth=3):
        super().__init__()
        layers = []
        in_c = 3

        for _ in range(depth):
            layers += [
                nn.Conv2d(in_c, channels, 3, padding=1),
                nn.BatchNorm2d(channels),
                nn.ReLU(),
                nn.Conv2d(channels, channels, 3, padding=1),
                nn.BatchNorm2d(channels),
                nn.ReLU(),
                nn.MaxPool2d(2)
            ]
            in_c = channels

        self.features = nn.Sequential(*layers)

        # dynamic flatten size
        with torch.no_grad():
            dummy = torch.zeros(1, 3, 32, 32)
            out = self.features(dummy)
            self.flatten_dim = out.numel()

        self.classifier = nn.Sequential(
            nn.Linear(self.flatten_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

c:\Users\getro\OneDrive\Desktop\School\e_graduate\Advanced Machine Learning\final-project\gap\.venv\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total = 0
    correct = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        pred = logits.argmax(1)
        correct += (pred == y).sum().item()
        total += y.size(0)
    return correct / total

def evaluate(model, loader):
    model.eval()
    total = 0
    correct = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            pred = logits.argmax(1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    return correct / total

In [16]:
from itertools import product

coarse_space = {
    "lr": [1e-4, 1e-3, 1e-2],
    "channels": [16, 32, 64],
    "depth": [1, 2, 3],
    "opt": ["sgd", "adam"]
}

def coarse_sensitivity(train_subset):
    results = []
    loader = DataLoader(train_subset, batch_size=128, shuffle=True)

    for lr, ch, depth, opt in product(*coarse_space.values()):
        #model = SmallCNN(channels=ch, depth=depth).to(device)
        model = BiggerCNN(channels=ch, depth=depth).to(device)
        
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.SGD(model.parameters(), lr=lr) if opt == "sgd" else optim.Adam(model.parameters(), lr=lr)

        acc = train_one_epoch(model, loader, criterion, optimizer)
        results.append(((lr, ch, depth, opt), acc))
        print((lr, ch, depth, opt), acc)

    return sorted(results, key=lambda x: x[1], reverse=True)

In [17]:
subset_idx = np.random.choice(len(train_data), 5000, replace=False)
train_subset = Subset(train_data, subset_idx)

coarse_results = coarse_sensitivity(train_subset)

(0.0001, 16, 1, 'sgd') 0.1114
(0.0001, 16, 1, 'adam') 0.2464
(0.0001, 16, 2, 'sgd') 0.1036
(0.0001, 16, 2, 'adam') 0.2052
(0.0001, 16, 3, 'sgd') 0.1078
(0.0001, 16, 3, 'adam') 0.135
(0.0001, 32, 1, 'sgd') 0.1036
(0.0001, 32, 1, 'adam') 0.2898
(0.0001, 32, 2, 'sgd') 0.1106
(0.0001, 32, 2, 'adam') 0.2454
(0.0001, 32, 3, 'sgd') 0.1102
(0.0001, 32, 3, 'adam') 0.185
(0.0001, 64, 1, 'sgd') 0.121
(0.0001, 64, 1, 'adam') 0.3098
(0.0001, 64, 2, 'sgd') 0.0934
(0.0001, 64, 2, 'adam') 0.275
(0.0001, 64, 3, 'sgd') 0.1002
(0.0001, 64, 3, 'adam') 0.2298
(0.001, 16, 1, 'sgd') 0.1106
(0.001, 16, 1, 'adam') 0.2904
(0.001, 16, 2, 'sgd') 0.1024
(0.001, 16, 2, 'adam') 0.2882
(0.001, 16, 3, 'sgd') 0.1092
(0.001, 16, 3, 'adam') 0.242
(0.001, 32, 1, 'sgd') 0.1394
(0.001, 32, 1, 'adam') 0.2338
(0.001, 32, 2, 'sgd') 0.1112
(0.001, 32, 2, 'adam') 0.2804
(0.001, 32, 3, 'sgd') 0.1138
(0.001, 32, 3, 'adam') 0.2872
(0.001, 64, 1, 'sgd') 0.1752
(0.001, 64, 1, 'adam') 0.2126
(0.001, 64, 2, 'sgd') 0.1304
(0.001, 64, 2,

In [18]:
def derive_refined_space(coarse_results, top_frac=0.2):
    # coarse_results is: [((lr, channels, depth, opt), acc), ...]
    top_k = int(len(coarse_results) * top_frac)

    # take top-performing configs
    top_configs = [cfg for (cfg, acc) in coarse_results[:top_k]]

    # unpack each hyperparameter dimension
    lrs      = [cfg[0] for cfg in top_configs]
    channels = [cfg[1] for cfg in top_configs]
    depths   = [cfg[2] for cfg in top_configs]
    opts     = [cfg[3] for cfg in top_configs]

    # build refined search space
    refined_space = {
        "lr": (min(lrs), max(lrs)),                 # continuous range
        "channels": sorted(set(channels)),          # categorical
        "depth": sorted(set(depths)),               # categorical
        "opt": sorted(set(opts))                    # categorical
    }

    return refined_space

In [19]:
refined_space = derive_refined_space(coarse_results)
print(refined_space)

{'lr': (0.0001, 0.01), 'channels': [16, 32, 64], 'depth': [1, 2, 3], 'opt': ['adam', 'sgd']}


In [20]:
import random

#refined_space = {
#    "lr": (5e-4, 5e-3),
#    "channels": [32, 48, 64],
#    "depth": [2, 3],
#    "opt": ["adam"]
#}

def sample_config():
    return {
        "lr": 10 ** np.random.uniform(np.log10(refined_space["lr"][0]),
                                      np.log10(refined_space["lr"][1])),
        "channels": random.choice(refined_space["channels"]),
        "depth": random.choice(refined_space["depth"]),
        "opt": "adam"
    }

In [21]:
def cross_val_score(config, k=3):
    kf = KFold(n_splits=k, shuffle=True, random_state=42)
    scores = []

    for train_idx, val_idx in kf.split(train_subset):
        train_loader = DataLoader(Subset(train_subset, train_idx), batch_size=128, shuffle=True)
        val_loader   = DataLoader(Subset(train_subset, val_idx), batch_size=128)

        #model = SmallCNN(config["channels"], config["depth"]).to(device)
        model = BiggerCNN(config["channels"], config["depth"]).to(device)
        
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=config["lr"])

        train_one_epoch(model, train_loader, criterion, optimizer)
        scores.append(evaluate(model, val_loader))

    return np.mean(scores)

In [22]:
best_score = 0
best_config = None

for _ in range(30):  # 30 trials
    cfg = sample_config()
    score = cross_val_score(cfg)
    print(cfg, score)

    if score > best_score:
        best_score = score
        best_config = cfg

print("Best:", best_config, best_score)

{'lr': 0.0004933533662821737, 'channels': 16, 'depth': 3, 'opt': 'adam'} 0.1773875717053468
{'lr': 0.00012306995135722757, 'channels': 64, 'depth': 1, 'opt': 'adam'} 0.32140330637353925
{'lr': 0.00028973837968351764, 'channels': 32, 'depth': 2, 'opt': 'adam'} 0.2208087794205865
{'lr': 0.0007571150901275153, 'channels': 16, 'depth': 2, 'opt': 'adam'} 0.18199961448286572
{'lr': 0.00010156698128790809, 'channels': 64, 'depth': 1, 'opt': 'adam'} 0.3125946239323564
{'lr': 0.00012885840451288473, 'channels': 64, 'depth': 2, 'opt': 'adam'} 0.2203975771472356
{'lr': 0.000704054807689306, 'channels': 16, 'depth': 2, 'opt': 'adam'} 0.17319909367466244
{'lr': 0.00019129813718561612, 'channels': 64, 'depth': 1, 'opt': 'adam'} 0.2946042244132206
{'lr': 0.0008024605282302414, 'channels': 64, 'depth': 2, 'opt': 'adam'} 0.2666023818045515
{'lr': 0.0007691531347764216, 'channels': 64, 'depth': 3, 'opt': 'adam'} 0.20100277663554925
{'lr': 0.0022566299147636854, 'channels': 64, 'depth': 1, 'opt': 'adam'}

In [23]:
#final_model = SmallCNN(best_config["channels"], best_config["depth"]).to(device)
final_model = BiggerCNN(best_config["channels"], best_config["depth"]).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(final_model.parameters(), lr=best_config["lr"])

train_loader = DataLoader(train_data, batch_size=128, shuffle=True)
test_loader  = DataLoader(test_data, batch_size=128)

for epoch in range(20):
    train_acc = train_one_epoch(final_model, train_loader, criterion, optimizer)
    test_acc  = evaluate(final_model, test_loader)
    print(epoch, train_acc, test_acc)

torch.save(final_model.state_dict(), "../saved_models/cifar10_classifier.pth")

0 0.48044 0.5725
1 0.6073 0.632
2 0.66208 0.6711
3 0.69678 0.685
4 0.7226 0.6912
5 0.74294 0.7018
6 0.7655 0.6965
7 0.783 0.7216
8 0.80322 0.7123
9 0.82102 0.7185
10 0.832 0.7178
11 0.8484 0.7249
12 0.86456 0.7296
13 0.87686 0.7288
14 0.89124 0.7279
15 0.90012 0.7272
16 0.9108 0.7326
17 0.91878 0.7299
18 0.92346 0.7235
19 0.93226 0.7364
